In [4]:
# Run first on Google Colab
!pip install qutip -q

# §8 Outlook — Quantum Walks

**Course:** Introductory Quantum Computing — Summer School

## Learning objectives
- Simulate a continuous-time quantum walk on a cycle graph
- Contrast the quantum spreading rate with the classical random walk
- Observe interference patterns absent in classical diffusion


In [5]:
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
from scipy.linalg import expm
from math import gcd

print(f"QuTiP {qt.__version__}")

# ── Helpers (carried over from previous notebooks) ────────────────────────────
def qft_matrix(n):
    N = 2**n
    omega = np.exp(2j * np.pi / N)
    F = np.array([[omega**(j*k) / np.sqrt(N) for k in range(N)] for j in range(N)])
    return qt.Qobj(F, dims=[[2]*n, [2]*n])

def apply_to_qubit(gate, n, qubit):
    ops = [qt.qeye(2)] * n
    ops[qubit] = gate
    return qt.tensor(ops)

def controlled_u(U, n, ctrl, tgt):
    P0 = qt.basis(2,0) * qt.basis(2,0).dag()
    P1 = qt.basis(2,1) * qt.basis(2,1).dag()
    ops0 = [qt.qeye(2)] * n;  ops0[ctrl] = P0
    ops1 = [qt.qeye(2)] * n;  ops1[ctrl] = P1;  ops1[tgt] = U
    return qt.tensor(ops0) + qt.tensor(ops1)

def swap_gate(n, i, j):
    N = 2**n
    mat = np.zeros((N, N), dtype=complex)
    for k in range(N):
        bits = list(format(k, f'0{n}b'))
        bits[i], bits[j] = bits[j], bits[i]
        mat[int(''.join(bits), 2), k] = 1.0
    return qt.Qobj(mat, dims=[[2]*n, [2]*n])

def qft_circuit(n):
    U = qt.tensor([qt.qeye(2)] * n)
    H1 = qt.gates.hadamard_transform(1)
    for i in range(n):
        U = apply_to_qubit(H1, n, i) * U
        for j in range(i+1, n):
            m = j - i + 1
            Rm = qt.Qobj(np.diag([1.0, np.exp(2j*np.pi/2**m)]))
            U = controlled_u(Rm, n, j, i) * U
    for i in range(n // 2):
        U = swap_gate(n, i, n-1-i) * U
    return U

def qpe(U_system, eigenstate, t_clock):
    sys_dim   = U_system.shape[0]
    sys_n     = int(np.round(np.log2(sys_dim)))
    sys_dims  = [[2]*sys_n, [2]*sys_n]

    zero = qt.basis(2, 0)
    H1   = qt.gates.hadamard_transform(1)
    I_sys = qt.qeye(sys_dim);  I_sys.dims = sys_dims

    clock_zero = qt.tensor([zero]*t_clock)
    psi = qt.tensor(clock_zero, eigenstate)

    Ht = qt.gates.hadamard_transform(t_clock)
    H_full = qt.tensor(Ht, I_sys)
    total_n = t_clock + sys_n
    H_full.dims = [[2]*total_n, [2]*total_n]
    psi = H_full * psi

    for j in range(t_clock):
        power = t_clock - 1 - j
        U_pow = U_system.copy()
        for _ in range(power):
            U_pow = U_pow * U_pow

        P0 = qt.basis(2,0) * qt.basis(2,0).dag()
        P1 = qt.basis(2,1) * qt.basis(2,1).dag()
        ops0 = [qt.qeye(2)]*t_clock;  ops0[j] = P0
        ops1 = [qt.qeye(2)]*t_clock;  ops1[j] = P1
        clock_P0 = qt.tensor(ops0)
        clock_P1 = qt.tensor(ops1)

        gate = qt.tensor(clock_P0, I_sys) + qt.tensor(clock_P1, U_pow)
        gate.dims = [[2]*total_n, [2]*total_n]
        psi = gate * psi

    QFT_dag = qft_circuit(t_clock).dag()
    QFT_full = qt.tensor(QFT_dag, I_sys)
    QFT_full.dims = [[2]*total_n, [2]*total_n]
    psi = QFT_full * psi

    return psi

def measure_clock(psi_out, t_clock, sys_n, n_shots=2048):
    N_clock = 2**t_clock
    N_sys   = 2**sys_n

    amps = psi_out.full().flatten()
    probs_clock = np.zeros(N_clock)
    for k in range(N_clock):
        for s in range(N_sys):
            probs_clock[k] += abs(amps[k * N_sys + s])**2

    outcomes = np.random.choice(N_clock, size=n_shots, p=probs_clock)
    counts = {k: int(np.sum(outcomes == k)) for k in range(N_clock) if np.sum(outcomes == k) > 0}
    return counts, probs_clock

print("QPE helpers defined.")


QuTiP 5.3.0
QPE helpers defined.


---
## Optional: §8 Outlook — Continuous-Time Quantum Walk

A **quantum walk** on a graph $G$ with adjacency matrix $A_G$ evolves as $e^{-iA_G t}$.
This is Hamiltonian simulation with $H = A_G$ — the foundation of several quantum algorithms.


In [ ]:
# ── Continuous-time quantum walk on cycle C_6 ─────────────────────────────────
n_walk = 6   # vertices of cycle
A_cycle = np.zeros((n_walk, n_walk))
for i in range(n_walk):
    A_cycle[i, (i+1) % n_walk] = 1
    A_cycle[(i+1) % n_walk, i] = 1
H_walk = qt.Qobj(A_cycle)

# The eigenvalues of the cycle C_n are 2cos(2πk/n), k=0..n-1.
# For n=6 these happen to be the integers {2, 1, -1, -2, -1, 1}, so e^{-iAt} is
# EXACTLY periodic with period T = 2π. This integer spectrum is special to C_6
# (and C_1..C_4); for general n the eigenvalues are irrational and the walk is
# only quasi-periodic. We pick C_6 precisely to get a clean, exactly-recurrent picture.
print("Eigenvalues of A(C_6):", np.round(sorted(np.linalg.eigvalsh(A_cycle)), 3))

# Start at vertex 0
psi_walk = qt.basis(n_walk, 0)

# Evolve and record the probability distribution over two periods (0 to 4π)
t_vals = np.linspace(0, 4*np.pi, 200)
probs_walk = np.zeros((len(t_vals), n_walk))

for ti, t_val in enumerate(t_vals):
    U_t = (-1j * H_walk * t_val).expm()
    psi_t = U_t * psi_walk
    probs_walk[ti] = np.abs(psi_t.full().flatten())**2

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# interpolation='none': vertices are discrete, so show sharp cells (no smooth blending)
im = axes[0].imshow(probs_walk.T, aspect='auto', origin='lower', interpolation='none',
                     extent=[0, 4*np.pi, -0.5, n_walk-0.5], cmap='hot')
axes[0].set_xlabel('time t'); axes[0].set_ylabel('vertex')
axes[0].set_title('Quantum walk on C₆: P(vertex|t)')
axes[0].set_yticks(range(n_walk))
axes[0].axvline(2*np.pi, color='cyan', lw=0.8, ls='--', label='T=2π')
axes[0].legend(fontsize=8); fig.colorbar(im, ax=axes[0])

# Compare with classical random walk at t = pi (clearest interference pattern)
# Quantum interference at t=pi: P = [1/9, 0, 4/9, 0, 4/9, 0]
# (destructive at odd-distance vertices, constructive at even-distance)
L = 2*np.eye(n_walk) - A_cycle   # graph Laplacian, d/dt p = -L p
p0_cl = np.zeros(n_walk); p0_cl[0] = 1.0

t_compare = np.pi
idx_compare = np.argmin(np.abs(t_vals - t_compare))
P_classical_compare = expm(-L * t_compare) @ p0_cl

axes[1].bar(np.arange(n_walk) - 0.2, probs_walk[idx_compare], width=0.4,
            alpha=0.8, label=f'quantum (t=π)')
axes[1].bar(np.arange(n_walk) + 0.2, P_classical_compare, width=0.4,
            alpha=0.6, label=f'classical (t=π)', color='orange')
axes[1].set_xlabel('vertex'); axes[1].set_ylabel('probability')
axes[1].set_title('Quantum vs classical walk at t=π')
axes[1].set_xticks(range(n_walk))
axes[1].legend()
plt.tight_layout(); plt.show()
print("At t=π: quantum walk has P=[1/9,0,4/9,0,4/9,0] — destructive interference at")
print("odd-distance vertices (1,3,5) and constructive at even-distance (2,4).")
print("Classical walk has diffused to near-uniform. The quantum walk is not simply")
print('faster; it oscillates with period 2π (returning to |0⟩ exactly at t=2π)')
print("while the classical walk converges monotonically to the uniform distribution.")

---
## Summary

| Topic | Key result |
|-------|-----------|
| CTQW generator | $U(t) = e^{-iAt}$, $H = A$ (adjacency matrix); unitary evolution on graph vertices |
| Periodicity on $C_6$ | Cycle eigenvalues are $2\cos(2\pi k/n)$; for $n=6$ these are integers $\pm1,\pm2\Rightarrow$ period $T=2\pi$ (special to $C_6$; general $C_n$ is only quasi-periodic) |
| Quantum interference | At $t=\pi$: $P = [\tfrac{1}{9},0,\tfrac{4}{9},0,\tfrac{4}{9},0]$ — destructive at odd-distance vertices, constructive at even-distance |
| vs. classical | Classical CTRW ($\dot{p}=-Lp$) converges monotonically to uniform; quantum walk oscillates, never mixes |

---
*End of course — you now have working implementations of QFT, QPE, Shor's algorithm, Grover search, Hamiltonian simulation, HHL, and continuous-time quantum walks.*